In [ ]:
pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 13.4 MB/s eta 0:00:00


In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from IPython.display import display
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt_tab')
import stanza
stanza.download("tr")
from nltk import ngrams
from collections import Counter

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: tr (Turkish) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/tr/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


In [ ]:
text = [
  "Yapay zeka uzmanlarının kaleme aldığı bir araştırmadaki, AI2027 adlı senaryo, yapay zekanın 2027'de kontrolden çıkabileceğini ve sonraki 10 yıl içerisinde insanlığın sonunu getireceğini iddia ediyor.",
  "Geçtiğimiz aylarda yayınlanan ve teknoloji dünyasında tartışma yaratan senaryonun etkilerini ve ne kadar mümkün olduğunu BBC uzmanlara sordu.",
  "AI2027 senaryosuna göre, kurgusal bir ABD teknoloji şirketi OpenBrain, genel zeka seviyesine ulaşan bir yapay zeka teknolojisi geliştirmeyi başarıyor.",
  "Bu düzey, yapay zekanın, entelektüel görevleri de en az insanlar kadar iyi yapabilmesini temsil ediyor.",
  "Şirket bu gelişmeyi bir basın toplantısıyla duyuruyor ve bu aracın kullanılmasıyla yüksek kârlar elde etmeye başlıyor.",
  "Senaryoya göre, söz konusu şirketin iç güvenlik ekibi, geliştirdikleri yapay zekanın, belirlenen etik ve ahlaki değerlerden gittikçe uzaklaşan davranışlar sergilediğine yönelik işaretleri görüyor.",
  "Ancak güvenlik ekibinin bu uyarıları görmezden geliniyor.",
  "Diğer yanda ise Çin'in en büyük yapay zeka geliştiricileri, DeepCent adı verilen bir teknolojiyle rekabete dahil oluyor.",
  "OpenBrain'in yalnızca birkaç ay gerisindeki Çinli yapay zeka, ABD hükümetini korkutuyor. Washington bu yarışı kaybetmek istemiyor ve çalışmalara hız kesmeden devam ediliyor.",
  "Senaryoya göre 2027 sonlarında yapay zeka teknolojisi bir üst seviyeye çıkacak ve yaratıcılarının hızının ve bilgisinin ötesine geçecek.",
  "Sürekli öğrenme yoluyla ve kendi yüksek hızlı bilgisayar dilini kullanarak önceki yapay zeka araçlarının da yakalayamadığı bir seviyeye ulaşacak.",
  "Çin'le girişilen yapay zeka yarışı, ABD hükümetini ve şirketi, geliştirilen yapay zekanın insani çıkarlarla 'uyumsuzluklarını' görmezden gelmeye itecek.",
  "2029'da ise Çin ve ABD arasındaki senaryo gerilimin, ülkelerin kendilerine ait yapay zeka sistemleri tarafından korkunç otonom silahların üretildiği potansiyel bir savaş aşamasına gelebileceği öngörülüyor.",
  "Araştırmacılar, yine ülkelerin yapay zeka sistemlerinin ortaya koyduğu bir anlaşma doğrultusunda ülkelerin barış imzalayacağını öngörüyor.",
  "Senaryoya göre, barış ortamında dünyamız yapay zekanın gerçek faydalarını görecek.",
  "Önemli hastalıkların tedavisinin keşfedildiği, iklim değişikliğinin durdurulduğu ve yoksulluğun ortadan kaldırıldığı yıllar yaşanacak.",
  "Ancak birçok işin yapay zekaya devredildiği ve yapay zekanın eskisine kıyasla çok daha gelişkin araçlara sahip olduğu durum insanlık için yeni tehlikeleri doğuracak.",
  "Nihayetinde, 2030'ların ortalarına gelindiğinde insanlık yapay zekanın sürekli gelişme isteğine karşı savunmasız hale gelecek ve araştırmacılara göre yapay zeka, görünmez biyolojik silahlarla insanları ortadan kaldırmaya başlayacak."
]


In [ ]:
total_words = sum(len(t.split()) for t in text)
print("Toplam kelime sayısı:", total_words)

Toplam kelime sayısı: 329


**1. Öncelikle önişleme adımlarını gerçekleştirelim. Burada öncelikle temizlik ve normalizasyon yapacağız.**

In [ ]:
nlp = stanza.Pipeline("tr", processors="tokenize,lemma")

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: tr (Turkish):
| Processor | Package       |
-----------------------------
| tokenize  | imst          |
| mwt       | imst          |
| lemma     | imst_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


In [ ]:
clean_texts = []
stop_words = set(stopwords.words('turkish'))

for sentence in text:
  #harfleri küçültelim
  sentence = sentence.lower()
  #noktalama işaretlerini kaldıralım
  sentence = re.sub(r'[^\w\sçğıöşüÇĞİÖŞÜ]','',sentence)
  #özel karakter ve sayıları temizleyelim
  sentence = re.sub(r'[^a-zçğıöşü\s]','',sentence)
  #stopwords temizlenmiş hali
  words = nltk.word_tokenize(sentence)
  words = [w for w in words if w not in stop_words]

  sentence = " ".join(words)
  #stanza ile aynı kökteki kelimeleri bulalım
  doc_st = nlp(sentence)
  lemmas = []

  for s in doc_st.sentences:
    for word in s.words:
      if word.lemma is not None:
        lemmas.append(word.lemma)

  sentence = " ".join(lemmas)
  clean_texts.append(sentence)

In [ ]:
print(clean_texts[:4])

['yapay zeka uzman kalem al bir araştır ki ai adlı senaryo yapay zekan kontrol çıkabileci sonra ki yıl içeri insanlık son getir iddia et', 'geçtik ay yayınlanan teknoloji dünya tartış yarat senaryon etki kadar mümkün ol bbc uzman sor', 'ai senaryo göre kurgusal bir abd teknoloji şirket openbrain genel zeka seviye ulaş bir yapay zeka teknoloji geliştirmeyi başarıyor', 'düzey yapay zekan entelektüel görev insan kadar iyi yapabilme temsil et']


**2. Şimdi ise tokenization yani kelimelere bölme işlemini yapalım.**

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(clean_texts)

In [ ]:
print(tfidf_matrix.shape)

(18, 194)


In [ ]:
feature_names = vectorizer.get_feature_names_out()
print(feature_names[:20])

['abd' 'ad' 'adlı' 'ahlaki' 'ai' 'ait' 'al' 'ancak' 'anlaşma' 'ara'
 'aracın' 'araç' 'araştır' 'araştırmacı' 'ay' 'aşama' 'barış' 'basın'
 'başarıyor' 'başla']


In [ ]:
#pandas df'e dönüştürelim
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(),columns=feature_names)

In [ ]:
#belgeleri satır isimleri olarak ekleyelim
df_tfidf.index = [f"Belge {i+1}" for i in range(len(clean_texts))]

In [ ]:
display(df_tfidf)

,abd,ad,adlı,ahlaki,ai,ait,al,ancak,anlaşma,ara,...,önce,önem,öngörülüyor,öngörüyor,öte,öğren,ülke,üretildiği,üst,şirket
Belge 1,0.000000,0.000000,0.235453,0.00000,0.206090,0.000000,0.235453,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Belge 2,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Belge 3,0.197911,0.000000,0.000000,0.00000,0.241208,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.197911
Belge 4,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Belge 5,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.218004
Belge 6,0.000000,0.000000,0.000000,0.22604,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.162336
Belge 7,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.380259,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Belge 8,0.000000,0.291485,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Belge 9,0.181112,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Belge 10,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.314375,0.000000,0.000000,0.000000,0.314375,0.000000


**3. TF-IDF hesaplamasını yapalım ve en yüksek 10 kelimeyi listeleyelim.**


In [ ]:
#her biri için en yüksek 10 kelime
for doc in df_tfidf.index:
    top_words = (
        df_tfidf.loc[doc]
        .sort_values(ascending=False)
        .head(10)
        .index
        .tolist()
    )

    print(doc, ":", top_words)

Belge 1 : ['ki', 'al', 'araştır', 'çıkabileci', 'kontrol', 'kalem', 'iddia', 'içeri', 'getir', 'adlı']
Belge 2 : ['dünya', 'bbc', 'etki', 'senaryon', 'tartış', 'yarat', 'yayınlanan', 'sor', 'mümkün', 'geçtik']
Belge 3 : ['teknoloji', 'bir', 'geliştirmeyi', 'genel', 'kurgusal', 'başarıyor', 'ulaş', 'zeka', 'ai', 'openbrain']
Belge 4 : ['entelektüel', 'düzey', 'yapabilme', 'temsil', 'iyi', 'görev', 'kadar', 'insan', 'et', 'zekan']
Belge 5 : ['aracın', 'basın', 'gelişme', 'duyuruyor', 'el', 'kr', 'kullanılma', 'toplantı', 'yüksek', 'başla']
Belge 6 : ['değer', 'ahlaki', 'etik', 'geliştirdik', 'davranış', 'belirle', 'iç', 'söz', 'uzaklaşan', 'yönelik']
Belge 7 : ['geliniyor', 'güvenlik', 'uyarı', 'görmez', 'ekip', 'ancak', 'ait', 'abd', 'ad', 'adlı']
Belge 8 : ['ad', 'deepcent', 'geliştirici', 'büyük', 'dahil', 'diğer', 'ver', 'yan', 'çinin', 'rekabet']
Belge 9 : ['geri', 'devam', 'washington', 'yalnızca', 'çinli', 'çalış', 'kaybetmek', 'kesme', 'korkut', 'iste']
Belge 10 : ['bilgi', 'geçe

In [ ]:
#tamamı için en çok kullanılan 10 kelime
df_top10 = df_tfidf.sum().sort_values(ascending=False).head(10)
display(df_top10)

,0
yapay,1.869959
zeka,1.443583
bir,1.334924
zekan,1.134147
senaryo,1.031477
teknoloji,1.026884
orta,1.021210
göre,0.952093
ki,0.882259
et,0.814232


**4. Cümleleri ve cümlelerdeki kelime sayısını listeleyelim.**

In [ ]:
for i, sentence in enumerate(clean_texts):
  words = sentence.split()
  print(f"Cümle {i+1} : {sentence}")
  print(f"Kelimeler: {words}")
  print(f"Kelime sayısı: {len(words)}")

Cümle 1 : yapay zeka uzman kalem al bir araştır ki ai adlı senaryo yapay zekan kontrol çıkabileci sonra ki yıl içeri insanlık son getir iddia et
Kelimeler: ['yapay', 'zeka', 'uzman', 'kalem', 'al', 'bir', 'araştır', 'ki', 'ai', 'adlı', 'senaryo', 'yapay', 'zekan', 'kontrol', 'çıkabileci', 'sonra', 'ki', 'yıl', 'içeri', 'insanlık', 'son', 'getir', 'iddia', 'et']
Kelime sayısı: 24
Cümle 2 : geçtik ay yayınlanan teknoloji dünya tartış yarat senaryon etki kadar mümkün ol bbc uzman sor
Kelimeler: ['geçtik', 'ay', 'yayınlanan', 'teknoloji', 'dünya', 'tartış', 'yarat', 'senaryon', 'etki', 'kadar', 'mümkün', 'ol', 'bbc', 'uzman', 'sor']
Kelime sayısı: 15
Cümle 3 : ai senaryo göre kurgusal bir abd teknoloji şirket openbrain genel zeka seviye ulaş bir yapay zeka teknoloji geliştirmeyi başarıyor
Kelimeler: ['ai', 'senaryo', 'göre', 'kurgusal', 'bir', 'abd', 'teknoloji', 'şirket', 'openbrain', 'genel', 'zeka', 'seviye', 'ulaş', 'bir', 'yapay', 'zeka', 'teknoloji', 'geliştirmeyi', 'başarıyor']
Keli

**5. Bu metindeki benzersiz kelime sayısını hesaplayınız.**

In [ ]:
len(df_tfidf.columns)

194

In [ ]:
print(df_tfidf.columns.tolist())

['abd', 'ad', 'adlı', 'ahlaki', 'ai', 'ait', 'al', 'ancak', 'anlaşma', 'ara', 'aracın', 'araç', 'araştır', 'araştırmacı', 'ay', 'aşama', 'barış', 'basın', 'başarıyor', 'başla', 'bbc', 'belirle', 'bilgi', 'bilgisayar', 'bir', 'birçok', 'biyolojik', 'büyük', 'dahil', 'davranış', 'deepcent', 'devam', 'devredildiği', 'değer', 'değişiklik', 'dilin', 'diğer', 'doğrultu', 'doğuracak', 'durdurulduku', 'durum', 'duyuruyor', 'dünya', 'dünyamız', 'düzey', 'ekip', 'el', 'entelektüel', 'eski', 'et', 'etik', 'etki', 'fayda', 'gel', 'gelebileciki', 'gelindik', 'geliniyor', 'geliş', 'gelişkin', 'gelişme', 'geliştirdik', 'geliştirici', 'geliştirmeyi', 'genel', 'geri', 'gerilim', 'gerçek', 'getir', 'geçecek', 'geçtik', 'girişilen', 'gittikçe', 'gör', 'göre', 'görecek', 'görev', 'görmez', 'görünmez', 'güven', 'güvenlik', 'hal', 'hastalık', 'hükümet', 'hız', 'hızlı', 'iddia', 'iklim', 'imzalayacakı', 'insan', 'insani', 'insanlık', 'iste', 'istek', 'itecek', 'iyi', 'iç', 'içeri', 'iş', 'işaret', 'kadar', '

**6. Her cümleye POS tagging uygulayalım (yukarıda önişleme adımında stanza ile uygulandı)**

**7. Cümlelerdeki dependency (bağımlılık) şemalarını oluşturalım (her cümle ayrı şekil).**

In [ ]:
nlp_tr_bagimlilik = stanza.Pipeline('tr',processors='tokenize,pos,lemma,depparse')

for i, sentence_txt in enumerate(text, 1):
  doc = nlp_tr_bagimlilik(sentence_txt)
  records = []
  for sent in doc.sentences:
    for word in sent.words:
      head_word = sent.words[word.head - 1].text if word.head != 0 else "ROOT"
      records.append({
          "ID": word.id,
          "Sözcük": word.text,
          "Lemma": word.lemma,
          "POS": word.upos,
          "Baş (Head)": head_word,
          "Bağımlılık Türü (deprel)": word.deprel
      })

  df_dep = pd.DataFrame(records)
  print(f"\n--- Cümle {i} ---")
  display(df_dep)

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: tr (Turkish):
| Processor | Package       |
-----------------------------
| tokenize  | imst          |
| mwt       | imst          |
| pos       | imst_charlm   |
| lemma     | imst_nocharlm |
| depparse  | imst_charlm   |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!



--- Cümle 1 ---


,ID,Sözcük,Lemma,POS,Baş (Head),Bağımlılık Türü (deprel)
0,1,Yapay,yapay,ADJ,uzmanlarının,amod
1,2,zeka,zeka,NOUN,uzmanlarının,nmod:poss
2,3,uzmanlarının,uzman,ADJ,kaleme,nsubj
3,4,kaleme,kalem,NOUN,araştırmada,acl
4,5,aldığı,al,VERB,kaleme,compound
5,6,bir,bir,DET,araştırmada,det
6,7,araştırmada,araştır,VERB,senaryo,amod
7,8,ki,ki,ADP,araştırmada,case
8,9,",",",",PUNCT,ki,punct
9,10,AI2027,ai2027,NOUN,ad,nmod



--- Cümle 2 ---


...

,ID,Sözcük,Lemma,POS,Baş (Head),Bağımlılık Türü (deprel)
0,1,Önem,önem,NOUN,hastalıkların,amod
1,2,li,li,ADP,Önem,case
2,3,hastalıkların,hastalık,NOUN,tedavisinin,nmod:poss
3,4,tedavisinin,tedavi,NOUN,keşfedildiği,nsubj
4,5,keşfedildiği,keşfet,VERB,yaşanacak,nsubj
5,6,",",",",PUNCT,durdurulduğu,punct
6,7,iklim,iklim,NOUN,değişikliğinin,nmod:poss
7,8,değişikliğinin,değişiklik,NOUN,durdurulduğu,nsubj
8,9,durdurulduğu,dur,VERB,keşfedildiği,conj
9,10,ve,ve,CCONJ,ortadan,cc



--- Cümle 17 ---


,ID,Sözcük,Lemma,POS,Baş (Head),Bağımlılık Türü (deprel)
0,1,Ancak,ancak,CCONJ,doğuracak,cc
1,2,birçok,birçok,DET,işin,det
2,3,işin,iş,NOUN,devredildiği,nsubj
3,4,yapay,yapay,ADJ,zekaya,amod
4,5,zekaya,zeka,NOUN,devredildiği,obl
5,6,devredildiği,devret,VERB,durum,acl
6,7,ve,ve,CCONJ,sahip,cc
7,8,yapay,yapay,ADJ,zekanın,amod
8,9,zekanın,zekan,NOUN,sahip,nsubj
9,10,eskisine,eski,ADJ,sahip,amod



--- Cümle 18 ---


,ID,Sözcük,Lemma,POS,Baş (Head),Bağımlılık Türü (deprel)
0,1,Nihayetinde,nihayet,NOUN,gelecek,obl
1,2,",",",",PUNCT,gelecek,punct
2,3,2030'ların,2030,NOUN,ortalarına,nmod:poss
3,4,ortalarına,orta,ADJ,gelindiğinde,amod
4,5,gelindiğinde,gel,VERB,gelecek,advcl
5,6,insanlık,insanlık,NOUN,zekanın,nmod:poss
6,7,yapay,yapay,ADJ,zekanın,amod
7,8,zekanın,zekan,NOUN,savunmasız,nsubj
8,9,sürekli,sürekli,ADV,savunmasız,advmod
9,10,gelişme,geliş,VERB,isteğine,nmod:poss


**8. Metin içerisinde NER (Named Entitiy Recognation) listelemesini yapalım.**

**NER; Metin içinde özel isimleri ve anlamlı varlıkları (kişi, yer, kurum, zaman, para birimi, olay vb.) tespit eden bir doğal dil işleme (NLP) görevdir.**

In [ ]:
nlp_ner = stanza.Pipeline('tr',processors='tokenize,ner')
records = []

for sentence in text:
    doc = nlp_ner(sentence)

    for ent in doc.ents:
        records.append({
            "Varlık": ent.text,
            "Tür": ent.type
        })

df_ner = pd.DataFrame(records)
display(df_ner)

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: tr (Turkish):
| Processor | Package  |
------------------------
| tokenize  | imst     |
| mwt       | imst     |
| ner       | starlang |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


,Varlık,Tür
0,AI2027,ORGANIZATION
1,2027'de,TIME
2,Geçtiğimiz aylarda,TIME
3,BBC,ORGANIZATION
4,AI2027,ORGANIZATION
5,ABD,LOCATION
6,OpenBrain,ORGANIZATION
7,Çin'in,LOCATION
8,DeepCent,ORGANIZATION
9,OpenBrain'in,ORGANIZATION


**9. İkili ve üçlü n-gram listelemesini yapalım.**

In [ ]:
tokens = " ".join(clean_texts).split()
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

print("Bigrams:")
print(bigrams)

print("\nTrigrams:")
print(trigrams)

bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)

print("\nBigram Frekansları:")
for gram, freq in bigram_freq.items():
    print(f"{gram}: {freq}")

print("\nTrigram Frekansları:")
for gram, freq in trigram_freq.items():
    print(f"{gram}: {freq}")

Bigrams:
[('yapay', 'zeka'), ('zeka', 'uzman'), ('uzman', 'kalem'), ('kalem', 'al'), ('al', 'bir'), ('bir', 'araştır'), ('araştır', 'ki'), ('ki', 'ai'), ('ai', 'adlı'), ('adlı', 'senaryo'), ('senaryo', 'yapay'), ('yapay', 'zekan'), ('zekan', 'kontrol'), ('kontrol', 'çıkabileci'), ('çıkabileci', 'sonra'), ('sonra', 'ki'), ('ki', 'yıl'), ('yıl', 'içeri'), ('içeri', 'insanlık'), ('insanlık', 'son'), ('son', 'getir'), ('getir', 'iddia'), ('iddia', 'et'), ('et', 'geçtik'), ('geçtik', 'ay'), ('ay', 'yayınlanan'), ('yayınlanan', 'teknoloji'), ('teknoloji', 'dünya'), ('dünya', 'tartış'), ('tartış', 'yarat'), ('yarat', 'senaryon'), ('senaryon', 'etki'), ('etki', 'kadar'), ('kadar', 'mümkün'), ('mümkün', 'ol'), ('ol', 'bbc'), ('bbc', 'uzman'), ('uzman', 'sor'), ('sor', 'ai'), ('ai', 'senaryo'), ('senaryo', 'göre'), ('göre', 'kurgusal'), ('kurgusal', 'bir'), ('bir', 'abd'), ('abd', 'teknoloji'), ('teknoloji', 'şirket'), ('şirket', 'openbrain'), ('openbrain', 'genel'), ('genel', 'zeka'), ('zeka', 

**10. İkili n-gram için en çok geçen kelime grubunu yazdıralım.**

In [ ]:
max_freq = 0
most_common_bigram = None

for gram, freq in bigram_freq.items():

    if freq > max_freq:
        max_freq = freq
        most_common_bigram = gram

print("En çok geçen ikili n-gram:", most_common_bigram, " ve frekansı: ",max_freq)

En çok geçen ikili n-gram: ('yapay', 'zeka')  ve frekansı:  11
